In [14]:
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, ConfusionMatrixDisplay
from sklearn.metrics import confusion_matrix, classification_report
from imblearn.over_sampling import SMOTE
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.utils.class_weight import compute_class_weight
import xgboost as xgb
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV
import seaborn as sns
import matplotlib.pyplot as plt
import warnings
from sklearn.preprocessing import StandardScaler

df = pd.read_csv(r"C:\Users\lexgu\Downloads\UR3 CobotOps Data - Sheet1.csv")
columns_to_drop = ['Timestamp']
df = df.drop(columns = columns_to_drop, axis=1)

columns_to_drop = ['Num', 'Current_J0', 'Current_J1', 'Current_J2', 'Current_J3', 'Current_J4', 'Current_J5', 'Temperature_T0', 'Temperature_J1', 'Temperature_J2', 'Temperature_J3', 'Temperature_J4', 'Temperature_J5', 'cycle', 'Tool_current', 'Robot_ProtectiveStop']
df_1 = df.drop(columns = columns_to_drop, axis=1)
#clean data
df_cleaned = df_1.dropna()

#make speed values only positive
columns_to_modify = ['Speed_J0', 'Speed_J1', 'Speed_J2', 'Speed_J3', 'Speed_J4', 'Speed_J5']

for column in columns_to_modify:
    df_cleaned[column] = df_cleaned[column].abs()
df_cleaned = df_cleaned.loc[~(df_cleaned == 0).all(axis=1)]



C:\Users\lexgu\AppData\Local\Temp\ipykernel_14124\1499520097.py:36: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df_cleaned[column] = df_cleaned[column].abs()


In [15]:
X = df_cleaned.drop('grip_lost', axis = 1)

y = df_cleaned['grip_lost']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state = 42)

In [16]:
smote = SMOTE(random_state=42)
X_train, y_train = smote.fit_resample(X_train, y_train)

In [18]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

knn = KNeighborsClassifier()

param_grid = {
    'n_neighbors': np.arange(1, 31),
    'weights': ['uniform', 'distance'],
    'metric': ['euclidean', 'manhattan', 'minkowski']
}
grid_search = GridSearchCV(knn, param_grid, cv=2, scoring='f1_macro')
grid_search.fit(X_train, y_train)
print("Best Parameters:", grid_search.best_params_)

# Print the best parameters and the corresponding score

Best Parameters: {'metric': 'manhattan', 'n_neighbors': 1, 'weights': 'uniform'}


In [19]:
bestParams = {'metric': 'manhattan', 'n_neighbors': 1, 'weights': 'uniform'}
knn_tuned = KNeighborsClassifier(**bestParams)
knn_tuned.fit(X_train, y_train)
knn_tuned_pred = knn_tuned.predict(X_test)

In [21]:
knn_tuned_results = {
    'Accuracy': accuracy_score(y_test, knn_tuned_pred),
    'Precision': precision_score(y_test, knn_tuned_pred, zero_division=1),
    'Recall': recall_score(y_test, knn_tuned_pred, zero_division=1),
    'F1 Score': f1_score(y_test, knn_tuned_pred, zero_division=1)
}
print(knn_tuned_results)

{'Accuracy': 0.9012276785714286, 'Precision': 0.14492753623188406, 'Recall': 0.25316455696202533, 'F1 Score': 0.18433179723502305}
